# 01 — Dataset Exploration
**ML-Based 6G Channel Modeling and Beam Prediction | VIT Chennai | BECE311L**

This notebook loads the **DeepMIMO `asu_campus_3p5`** real ray-tracing dataset,
inspects its structure, and visualizes key distributions before training.

| Item | Value |
|------|-------|
| Scenario | `asu_campus_3p5` — outdoor campus, 3.5 GHz |
| BS antennas | 64-element ULA |
| UE antennas | 1 (single-antenna) |
| Total users | 131,931 |
| Valid users | ~85,157 (after filtering blocked) |
| Codebook size | N_b = 64 beams |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('dark_background')
matplotlib.rcParams.update({
    'font.family':    'DejaVu Sans',
    'axes.facecolor': '#0f1117',
    'figure.facecolor':'#0f1117',
    'axes.edgecolor': '#444',
    'grid.color':     '#2a2d3e',
    'grid.linestyle': '--',
    'grid.alpha':     0.5,
})

from src.dataset import (
    dft_codebook, load_deepmimo_v4,
    filter_valid_users, compute_beam_labels,
)
print('Imports OK')

## 1. Load DeepMIMO Dataset
Auto-downloads `asu_campus_3p5` if not already present (~29 MB).

In [ ]:
positions, channels = load_deepmimo_v4('asu_campus_3p5', N_t=64)

print(f'positions shape : {positions.shape}   (N_users, 3)')
print(f'channels shape  : {channels.shape}  (N_users, N_r, N_t)')
print(f'X range : [{positions[:,0].min():.1f}, {positions[:,0].max():.1f}] m')
print(f'Y range : [{positions[:,1].min():.1f}, {positions[:,1].max():.1f}] m')

## 2. Filter Blocked Users (Zero-power channels)

In [ ]:
channels_all, positions_all = channels.copy(), positions.copy()
channels, positions = filter_valid_users(channels, positions)
print(f'Valid users: {channels.shape[0]:,} / {channels_all.shape[0]:,} '
      f'({channels.shape[0]/channels_all.shape[0]*100:.1f}%)')

## 3. UE Position Map — All Users vs. Valid Users

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#0f1117')

for ax, pos, title, color in [
    (axes[0], positions_all, 'All Users (131,931)', '#7c83fd'),
    (axes[1], positions,     'Valid Users (non-blocked)', '#26de81'),
]:
    ax.set_facecolor('#0f1117')
    ax.scatter(pos[:,0], pos[:,1], s=0.5, alpha=0.4, c=color, linewidths=0)
    ax.set_xlabel('X (m)', fontsize=12, color='white')
    ax.set_ylabel('Y (m)', fontsize=12, color='white')
    ax.set_title(title, fontsize=13, color='white', pad=10)
    ax.tick_params(colors='white')
    ax.grid(True)

plt.suptitle('ASU Campus 3.5 GHz — UE Position Grid', fontsize=14, color='white', y=1.01)
plt.tight_layout()
plt.show()

## 4. Channel Norm Distribution

In [ ]:
norms = np.linalg.norm(channels.reshape(channels.shape[0], -1), axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(norms, bins=100, color='#f7b731', alpha=0.85, edgecolor='none')
ax.set_xlabel('Channel Frobenius Norm', fontsize=13)
ax.set_ylabel('Number of Users', fontsize=13)
ax.set_title('Channel Norm Distribution — asu_campus_3p5', fontsize=14)
ax.grid(True)
plt.tight_layout()
plt.show()
print(f'Mean: {norms.mean():.4f}  Std: {norms.std():.4f}  Max: {norms.max():.4f}')

## 5. DFT Codebook & Beam Label Distribution

In [ ]:
codebook = dft_codebook(N_t=64, N_b=64)
print(f'Codebook shape : {codebook.shape}')
print(f'Unit-norm      : {np.allclose(np.linalg.norm(codebook, axis=1), 1.0)}')

labels = compute_beam_labels(channels, codebook)
print(f'Label range    : [{labels.min()}, {labels.max()}]')
print(f'Unique beams   : {len(np.unique(labels))} / {codebook.shape[0]}')

In [ ]:
counts = np.bincount(labels, minlength=64)

fig, ax = plt.subplots(figsize=(12, 4))
cmap_vals = plt.cm.plasma(np.linspace(0.2, 0.9, 64))
ax.bar(np.arange(64), counts, color=cmap_vals, alpha=0.9, edgecolor='none')
ax.set_xlabel('Beam Index', fontsize=13)
ax.set_ylabel('Number of Users', fontsize=13)
ax.set_title('Optimal Beam Label Distribution — Real Channel Data (N_b=64)', fontsize=14)
ax.set_xlim(-1, 64)
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()
print(f'Most common beam  : {counts.argmax()} ({counts.max():,} users = {counts.max()/len(labels)*100:.1f}%)')
print(f'Least common beam : {counts.argmin()} ({counts.min():,} users)')

## 6. Spatial Beam Map — Optimal Labels
Each dot is a UE coloured by its optimal beam index. Clear spatial clusters
demonstrate the geometric structure the MLP learns.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

sc = ax.scatter(positions[:,0], positions[:,1],
                c=labels, cmap='hsv', vmin=0, vmax=63,
                s=4, alpha=0.7, linewidths=0)

cbar = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Optimal Beam Index', fontsize=12, color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.set_xlabel('X Position (m)', fontsize=13, color='white')
ax.set_ylabel('Y Position (m)', fontsize=13, color='white')
ax.set_title('Spatial Map of Optimal Beam Labels\nasu_campus_3p5 @ 3.5 GHz | 64-antenna ULA',
             fontsize=14, color='white', pad=12)
ax.tick_params(colors='white')
ax.grid(True)
plt.tight_layout()
plt.show()

## 7. Beam Pattern Polar Plots

In [ ]:
N_t = 64
theta_scan = np.linspace(-np.pi/2, np.pi/2, 500)
ant_idx    = np.arange(N_t)

fig, axes = plt.subplots(1, 4, figsize=(14, 4), subplot_kw={'projection': 'polar'})
sample_beams = [0, 16, 32, 48]
colors = ['#7c83fd', '#26de81', '#f7b731', '#ff6b6b']

for ax, beam_idx, color in zip(axes, sample_beams, colors):
    sin_theta_i = 2.0 * beam_idx / N_t - 1.0
    af = np.abs(np.sum(
        (1/np.sqrt(N_t)) * np.exp(1j * np.pi * ant_idx[None,:] *
        (np.sin(theta_scan)[:,None] - sin_theta_i)), axis=1))
    af_db = 20 * np.log10(af + 1e-10)
    af_db = np.clip(af_db - af_db.max(), -30, 0)

    ax.plot(theta_scan, af_db + 30, color=color, linewidth=1.5)
    ax.set_facecolor('#0f1117')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_thetamin(-90)
    ax.set_thetamax(90)
    deg = np.degrees(np.arcsin(np.clip(sin_theta_i, -1, 1)))
    ax.set_title(f'Beam {beam_idx}\n(θ={deg:.0f}°)', fontsize=10, color='white', pad=8)
    ax.tick_params(colors='white', labelsize=8)

fig.suptitle('DFT Codebook Beam Patterns (64-element ULA)', fontsize=13, color='white', y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Item | Value |
|------|-------|
| Scenario | `asu_campus_3p5` (outdoor, 3.5 GHz) |
| Total users | 131,931 |
| Valid users (non-blocked) | ~85,157 |
| Channel shape per user | (1, 64) — 1 RX × 64 TX antennas |
| Codebook size N_b | 64 beams |
| Unique beams in labels | 64 (all beams used) |
| Key observation | Clear spatial clusters in beam map → learnable structure |

**Next**: Run `python src/train.py` or open `02_results.ipynb`.